# Feature Exploration — Clinical Trial Prompts
### What concepts does the model activate when reasoning about clinical trials?

This notebook explores the attribution graphs produced by our cross-layer transcoder (CLT)
for 14 clinical trial prompts. Each graph shows which internal model features contributed
most to the model's prediction at the final token.

**Key finding from this run:** Pythia-410m, a general-purpose language model, relies almost
entirely on syntactic and domain-general features (auxiliary verbs, pronoun patterns, list
punctuation) rather than clinical concepts. This is expected — it motivates the next phase
of the project using MedGemma-27B trained on medical text.

**No model loading required** — all plots are derived from precomputed graph JSONs and feature labels.

In [ ]:
import json
import sys
from pathlib import Path
from collections import defaultdict

import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
from viz.graphs import plot_node_contributions, plot_layer_flow, summarize_graph

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

GRAPH_DIR   = Path("../frontend/graph_data")
LABELS_PATH = Path("../data/feature_labels.jsonl")
N_LAYERS    = 24


def load_labels(path: Path) -> dict[tuple[int, int], str]:
    labels = {}
    if not path.exists():
        return labels
    with open(path) as f:
        for line in f:
            obj = json.loads(line)
            labels[(int(obj["layer"]), int(obj["feature"]))] = obj["label"]
    return labels


def load_graph(slug: str) -> dict:
    return json.loads((GRAPH_DIR / f"{slug}.json").read_text())


def contributions_to_logit(graph: dict) -> tuple[list[dict], dict]:
    """Return (contrib_nodes, logit_node) where contrib_nodes are direct parents of the logit."""
    logit_node = next(n for n in graph["nodes"] if n["feature_type"] == "logit")
    node_by_id = {n["node_id"]: n for n in graph["nodes"]}
    type_map = {
        "cross layer transcoder": "feature",
        "embedding":              "embedding",
        "mlp reconstruction error": "error",
    }
    contrib_nodes = []
    for link in graph["links"]:
        if link["target"] == logit_node["node_id"]:
            src = node_by_id.get(link["source"])
            if src:
                contrib_nodes.append({
                    "label":        src["clerp"],
                    "contribution": link["weight"],
                    "type":         type_map.get(src["feature_type"], "feature"),
                    "layer":        src.get("layer"),
                })
    return contrib_nodes, logit_node


labels = load_labels(LABELS_PATH)
print(f"Loaded {len(labels)} feature labels")
print(f"Graph files: {len(list(GRAPH_DIR.glob('*.json')))}")

## Spotlight: What makes a patient eligible?

We start with `eligible_inclusion` — the model completes the sentence:
> *"The patient meets all inclusion criteria and is therefore ___"*

The target token is **" eligible"**. The chart below shows which features contribute most to
that prediction, and from which layers.

In [ ]:
graph = load_graph("eligible_inclusion")
contrib_nodes, logit_node = contributions_to_logit(graph)

print(f"Target token: {logit_node['clerp']}")
print(f"Direct contributors to logit: {len(contrib_nodes)}")

fig = plot_node_contributions(contrib_nodes, target_token=" eligible", topk=12)
plt.show()

In [ ]:
fig = plot_layer_flow(contrib_nodes, n_layers=N_LAYERS, target_token=" eligible")
plt.show()

## What kind of features are these?

The labels reveal the nature of the features driving the prediction.
We can categorize every CLT feature node across all 14 graphs.

In [ ]:
all_slugs = [p.stem for p in sorted(GRAPH_DIR.glob("*.json"))]
clinical_slugs = [s for s in all_slugs if s not in ("france_capital", "water_boil")]

# Collect all labeled CLT feature nodes across clinical graphs
all_feature_labels = []
for slug in clinical_slugs:
    g = load_graph(slug)
    for node in g["nodes"]:
        if node["feature_type"] == "cross layer transcoder":
            clerp = node["clerp"]
            if not clerp.startswith("L"):  # has a real label (not fallback "L0F12@3")
                all_feature_labels.append(clerp)

print(f"Total labeled CLT feature nodes across {len(clinical_slugs)} clinical graphs: {len(all_feature_labels)}")
print(f"\nSample labels:")
for lbl in sorted(set(all_feature_labels))[:20]:
    print(f"  {lbl}")

In [ ]:
# Count unique labels and how often each appears
from collections import Counter
label_counts = Counter(all_feature_labels)

print("Most frequently appearing feature labels across clinical graphs:\n")
for label, count in label_counts.most_common(20):
    print(f"  {count:3d}x  {label}")

## Which features are shared across multiple clinical graphs?

If the same feature fires across many different clinical prompts, it is a general-purpose
language feature reused by the model regardless of clinical context.
Features unique to one graph are more likely to be prompt-specific.

In [ ]:
# Map (layer, feature) → set of graphs it appears in
feature_to_graphs: dict[tuple[int, int], set[str]] = defaultdict(set)
feature_to_label:  dict[tuple[int, int], str] = {}

for slug in clinical_slugs:
    g = load_graph(slug)
    for node in g["nodes"]:
        if node["feature_type"] == "cross layer transcoder":
            key = (node["layer"], node["feature"])
            feature_to_graphs[key].add(slug)
            if not node["clerp"].startswith("L"):
                feature_to_label[key] = node["clerp"]

# Features appearing in 3+ graphs
shared = [
    (key, len(graphs), feature_to_label.get(key, f"L{key[0]}F{key[1]} (unlabeled)"))
    for key, graphs in feature_to_graphs.items()
    if len(graphs) >= 3
]
shared.sort(key=lambda x: x[1], reverse=True)

print(f"Features appearing in 3+ clinical graphs: {len(shared)}\n")
df = pd.DataFrame([
    {"Layer": k[0], "Feature": k[1], "Graphs": n, "Label": lbl}
    for k, n, lbl in shared
])
df

In [ ]:
# Visualise: how many features are shared vs unique?
from collections import Counter as C
share_counts = C(len(graphs) for graphs in feature_to_graphs.values())

fig, ax = plt.subplots(figsize=(7, 4))
xs = sorted(share_counts)
ax.bar(xs, [share_counts[x] for x in xs], color="#4C72B0")
ax.set_xlabel("Number of clinical graphs the feature appears in")
ax.set_ylabel("Number of features")
ax.set_xticks(xs)
ax.set_title("Feature sharing across 12 clinical graphs", fontweight="bold")
fig.tight_layout()
plt.show()

## Compare two graphs side-by-side

What does the model focus on differently for an *eligible* patient vs an *ineligible* one?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, slug, token in [
    (axes[0], "eligible_inclusion",    " eligible"),
    (axes[1], "ineligible_prior_therapy", " ineligible"),
]:
    g = load_graph(slug)
    nodes, logit = contributions_to_logit(g)
    top = sorted(nodes, key=lambda n: abs(n["contribution"]), reverse=True)[:10]

    type_colors = {"feature": "#4C72B0", "embedding": "#55A868", "error": "#C44E52"}
    colors = [type_colors.get(n["type"], "#888") for n in top]
    ax.barh(range(len(top)), [n["contribution"] for n in top], color=colors)
    ax.set_yticks(range(len(top)))
    ax.set_yticklabels([n["label"] for n in top], fontsize=9)
    ax.invert_yaxis()
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel("Contribution to logit")
    ax.set_title(f'→ "{token.strip()}"  ({slug})', fontweight="bold", fontsize=10)

fig.suptitle("Top contributing features: eligible vs ineligible", fontsize=12, fontweight="bold")
fig.tight_layout()
plt.show()

## What this tells us

**Pythia-410m reasons about clinical eligibility using surface syntax, not clinical knowledge.**

The features driving predictions like "eligible" or "ineligible" are things like:
- Auxiliary verbs and modal constructions
- Third-person pronouns and subject markers
- List punctuation and conjunctions
- General sentence completion patterns

This is the model detecting the *shape* of the sentence (a passive construction, a formal
enumeration, a consequence clause) rather than understanding what the criteria actually mean.

**This is the core motivation for the MedGemma phase.** A model pretrained on clinical text
should learn features like:
- ECOG performance status thresholds
- Prior therapy line counts
- Lab value ranges and toxicity grades
- Eligibility criterion structures

The CLT methodology is already working — we can read the internal representations.
The question is whether those representations carry clinical meaning.

---
*Next: apply this same pipeline to MedGemma-27B + MIMIC-IV clinical notes.*

In [ ]:
# Summary table: all graphs, target token, top contributing feature
rows = []
for slug in sorted(clinical_slugs):
    g = load_graph(slug)
    contrib_nodes, logit_node = contributions_to_logit(g)
    if not contrib_nodes:
        continue
    top = max(contrib_nodes, key=lambda n: abs(n["contribution"]))
    rows.append({
        "Graph": slug,
        "Target token": logit_node["clerp"].split('"')[1] if '"' in logit_node["clerp"] else "—",
        "Top feature": top["label"],
        "Contribution": f"{top['contribution']:+.3f}",
        "Type": top["type"],
    })

pd.DataFrame(rows)